In [1]:
# ===== CELL 1: install + GPU =====
# !pip -q install "transformers>=4.40" "datasets>=2.19" accelerate

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"

torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA L4


In [2]:
# ===== CELL 2: build datasets from HH-RLHF (train + held-out test) =====
import random, json
from pathlib import Path
from datasets import load_dataset, concatenate_datasets

SUBSETS = ["helpful-base"]   # add "harmless-base" etc. for the full HH objective

def load_split(split):
    parts = [load_dataset("Anthropic/hh-rlhf", data_dir=s, split=split) for s in SUBSETS]
    ds = parts[0]
    for p in parts[1:]:
        ds = concatenate_datasets([ds, p])
    return ds

train_data = load_split("train")
test_data  = load_split("test")

print("Train examples:", len(train_data))
print("Test examples :", len(test_data))

MARKER = "\n\nAssistant:"
def split_conversation(text):
    idx = text.rfind(MARKER)
    if idx == -1:
        return None, None
    return text[:idx].strip(), text[idx + len(MARKER):].strip()

text_sample = "Human: What is AI?\n\nAssistant: AI is artificial intelligence."
print(split_conversation(text_sample))

README.md:   0%|          | 0.00/5.77k [00:00<?, ?B/s]

helpful-base/train.jsonl.gz: reconstructing file:   0%|          |  0.00B / 16.2MB            

helpful-base/train.jsonl.gz: downloading bytes:           |  0.00B            

helpful-base/test.jsonl.gz: reconstructing file:   0%|          |  0.00B /  875kB            

helpful-base/test.jsonl.gz: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train examples: 43835
Test examples : 2354
('Human: What is AI?', 'AI is artificial intelligence.')


In [3]:
def clean(dataset):
    rows = []
    for ex in dataset:
        cp, cr = split_conversation(ex["chosen"])
        rp, rr = split_conversation(ex["rejected"])
        if cp is None or rp is None or not cr or not rr:
            continue
        rows.append({"prompt": cp, "chosen": cr, "rejected": rr})
    return rows

clean_train = clean(train_data)
clean_test  = clean(test_data)
print("Clean train:", len(clean_train), "| Clean test:", len(clean_test))

Clean train: 43783 | Clean test: 2350


In [4]:
random.seed(42)
random.shuffle(clean_train)

# three non-overlapping TRAIN subsets
N_SFT, N_RM, N_PPO = 300, 300, 150
required = N_SFT + N_RM + N_PPO
if len(clean_train) < required:
    scale = len(clean_train) / required
    N_SFT, N_RM, N_PPO = int(N_SFT*scale), int(N_RM*scale), int(N_PPO*scale)
    print(f"Shrunk to SFT={N_SFT} RM={N_RM} PPO={N_PPO}")

sft_raw = clean_train[:N_SFT]
rm_raw  = clean_train[N_SFT:N_SFT+N_RM]
ppo_raw = clean_train[N_SFT+N_RM:N_SFT+N_RM+N_PPO]

# held-out TEST set (never seen in any training stage)
N_RM_TEST, N_EVAL = 100, 20
rm_test_raw  = clean_test[:N_RM_TEST]
eval_prompts = clean_test[N_RM_TEST:N_RM_TEST+N_EVAL]

out = Path("hh_rlhf_instructgpt"); out.mkdir(exist_ok=True)

def save_jsonl(rows, name):
    path = out / name
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return path

save_jsonl([{"prompt": x["prompt"], "response": x["chosen"]} for x in sft_raw], "sft.jsonl")
save_jsonl([{"prompt": x["prompt"], "chosen": x["chosen"], "rejected": x["rejected"]} for x in rm_raw], "rm.jsonl")
save_jsonl([{"prompt": x["prompt"]} for x in ppo_raw], "ppo.jsonl")
save_jsonl([{"prompt": x["prompt"], "chosen": x["chosen"], "rejected": x["rejected"]} for x in rm_test_raw], "rm_test.jsonl")
save_jsonl([{"prompt": x["prompt"]} for x in eval_prompts], "eval_prompts.jsonl")

print(f"SFT={len(sft_raw)} RM={len(rm_raw)} PPO={len(ppo_raw)} "
      f"| RM_test={len(rm_test_raw)} eval={len(eval_prompts)}")

SFT=300 RM=300 PPO=150 | RM_test=100 eval=20


In [5]:
# ===== CELL 3: tokenizer + formatting =====
from transformers import AutoTokenizer

MODEL_NAME = "gpt2"
MAX_LEN = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def format_sft(prompt, response):
    return f"{prompt}\n\nAssistant: {response}{tokenizer.eos_token}"

def format_prompt_for_gen(prompt):
    return f"{prompt}\n\nAssistant:"

def format_rm_side(prompt, response):
    return f"{prompt}\n\nAssistant: {response}{tokenizer.eos_token}"

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
# ===== CELL 4: Stage 1 — SFT with RESPONSE-ONLY loss =====
import json, torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM

class SFTDataset(Dataset):
    def __init__(self, path):
        self.rows = [json.loads(l) for l in open(path)]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        # tokenize prompt (+ marker) and response separately so we know the split
        prompt_text = f'{r["prompt"]}\n\nAssistant:'
        prompt_ids  = tokenizer(prompt_text, truncation=True, max_length=MAX_LEN)["input_ids"]
        full_text   = format_sft(r["prompt"], r["response"])   # prompt + " response" + eos
        full_ids    = tokenizer(full_text, truncation=True, max_length=MAX_LEN)["input_ids"]
        prompt_len  = min(len(prompt_ids), len(full_ids))      # guard against truncation
        return full_ids, prompt_len

def sft_collate(batch):
    maxlen = max(len(ids) for ids, _ in batch)
    pad = tokenizer.pad_token_id
    input_ids, attn, labels = [], [], []
    for ids, prompt_len in batch:
        padn = maxlen - len(ids)
        input_ids.append(ids + [pad]*padn)
        attn.append([1]*len(ids) + [0]*padn)
        # mask BOTH the prompt tokens and the padding; keep only response tokens
        lab = [-100]*prompt_len + ids[prompt_len:] + [-100]*padn
        labels.append(lab)
    return torch.tensor(input_ids), torch.tensor(attn), torch.tensor(labels)

sft_dl = DataLoader(SFTDataset("hh_rlhf_instructgpt/sft.jsonl"),
                    batch_size=4, shuffle=True, collate_fn=sft_collate)

sft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
opt = torch.optim.AdamW(sft_model.parameters(), lr=5e-5)

sft_model.train()
EPOCHS = 3
for ep in range(EPOCHS):
    for step, (ids, attn, labels) in enumerate(sft_dl):
        ids, attn, labels = ids.to(device), attn.to(device), labels.to(device)
        loss = sft_model(input_ids=ids, attention_mask=attn, labels=labels).loss
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)
        opt.step()
        if step % 50 == 0:
            print(f"[SFT] epoch {ep} step {step}/{len(sft_dl)} loss {loss.item():.4f}")

sft_model.save_pretrained("sft_model"); tokenizer.save_pretrained("sft_model")
print("Saved sft_model/")

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


[SFT] epoch 0 step 0/75 loss 3.4489
[SFT] epoch 0 step 50/75 loss 2.6047
[SFT] epoch 1 step 0/75 loss 2.3030
[SFT] epoch 1 step 50/75 loss 2.2087
[SFT] epoch 2 step 0/75 loss 1.9515
[SFT] epoch 2 step 50/75 loss 1.3904


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved sft_model/


In [7]:
# ===== CELL 5: Stage 2 — Reward Model (pairwise Bradley-Terry) + test-set eval =====
import json, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

class RewardModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        h = self.transformer.config.hidden_size
        self.v_head = nn.Linear(h, 1, bias=False)
        nn.init.normal_(self.v_head.weight, std=1.0/np.sqrt(h+1))
    def forward(self, input_ids, attention_mask):
        hs = self.transformer(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        r = self.v_head(hs).squeeze(-1)               # (B, T)
        last = attention_mask.sum(1) - 1              # last real token
        b = torch.arange(input_ids.size(0), device=input_ids.device)
        return r[b, last]                             # (B,)

class RMDataset(Dataset):
    def __init__(self, path):
        self.rows = [json.loads(l) for l in open(path)]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        c = tokenizer(format_rm_side(r["prompt"], r["chosen"]),   truncation=True, max_length=MAX_LEN)
        j = tokenizer(format_rm_side(r["prompt"], r["rejected"]), truncation=True, max_length=MAX_LEN)
        return c["input_ids"], j["input_ids"]

def rm_collate(batch):
    pad = tokenizer.pad_token_id
    def pack(seqs):
        maxlen = max(len(s) for s in seqs)
        ids = torch.full((len(seqs), maxlen), pad, dtype=torch.long)
        att = torch.zeros((len(seqs), maxlen), dtype=torch.long)
        for i, s in enumerate(seqs):
            ids[i,:len(s)] = torch.tensor(s); att[i,:len(s)] = 1
        return ids, att
    c_ids, c_att = pack([b[0] for b in batch])
    r_ids, r_att = pack([b[1] for b in batch])
    return c_ids, c_att, r_ids, r_att

# Split the RM training file into train/val. Checkpoint selection uses VAL, so
# the held-out TEST set stays untouched until the final report (no test leakage
# into model selection).
from torch.utils.data import random_split
_rm_full = RMDataset("hh_rlhf_instructgpt/rm.jsonl")
_val_n   = max(1, int(0.1 * len(_rm_full)))
_train_n = len(_rm_full) - _val_n
rm_train_ds, rm_val_ds = random_split(
    _rm_full, [_train_n, _val_n],
    generator=torch.Generator().manual_seed(42))

rm_dl      = DataLoader(rm_train_ds, batch_size=4, shuffle=True,  collate_fn=rm_collate)
rm_val_dl  = DataLoader(rm_val_ds,   batch_size=4, shuffle=False, collate_fn=rm_collate)
rm_test_dl = DataLoader(RMDataset("hh_rlhf_instructgpt/rm_test.jsonl"),
                        batch_size=4, shuffle=False, collate_fn=rm_collate)

# Initialize the reward model's trunk from the SFT checkpoint when available
# (InstructGPT initializes the RM from the SFT model, not the base LM). The
# SFT dir stores an AutoModelForCausalLM whose transformer trunk is weight-
# compatible with the AutoModel trunk used here.
import os
RM_INIT = "sft_model" if os.path.isdir("sft_model") else MODEL_NAME
reward_model = RewardModel(RM_INIT).to(device)
print(f"[RM] initialized trunk from: {RM_INIT}")
opt = torch.optim.AdamW(reward_model.parameters(), lr=1e-5)

EPOCHS = 3

@torch.no_grad()
def rm_accuracy(dl):
    reward_model.eval()
    correct = tot = 0
    for c_ids, c_att, r_ids, r_att in dl:
        rc = reward_model(c_ids.to(device), c_att.to(device))
        rr = reward_model(r_ids.to(device), r_att.to(device))
        correct += (rc > rr).sum().item(); tot += rc.size(0)
    reward_model.train()
    return correct / tot

print(f"[RM] val accuracy before training: {rm_accuracy(rm_val_dl):.4f}")

best_acc = -1.0
for ep in range(EPOCHS):
    reward_model.train()
    running_loss = running_acc = n = 0
    for step, (c_ids, c_att, r_ids, r_att) in enumerate(rm_dl):
        c_ids, c_att, r_ids, r_att = c_ids.to(device), c_att.to(device), r_ids.to(device), r_att.to(device)
        rc = reward_model(c_ids, c_att)
        rr = reward_model(r_ids, r_att)
        loss = -torch.nn.functional.logsigmoid(rc - rr).mean()   # Bradley-Terry
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        opt.step()

        running_loss += loss.item()
        running_acc  += (rc > rr).float().mean().item()
        n += 1
        if step % 50 == 0:
            print(f"[RM] epoch {ep} step {step}/{len(rm_dl)} loss {loss.item():.4f} "
                  f"batch_acc {(rc > rr).float().mean().item():.3f}")

    val_acc = rm_accuracy(rm_val_dl)
    print(f"[RM] epoch {ep} done | train_loss {running_loss/n:.4f} "
          f"train_acc {running_acc/n:.3f} | val_acc {val_acc:.4f}")

    # Select the checkpoint on VALIDATION (not test) to avoid test leakage into
    # model selection. Test is reported once, after training, on the best model.
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({"state_dict": reward_model.state_dict(), "model_name": RM_INIT},
                   "reward_model.pt")
        print(f"[RM] new best val_acc {best_acc:.4f} -> saved reward_model.pt")

# Final unbiased estimate: reload best checkpoint, evaluate on held-out TEST once.
_best = torch.load("reward_model.pt", map_location=device)
reward_model.load_state_dict(_best["state_dict"])
final_test_acc = rm_accuracy(rm_test_dl)
print(f"[RM] training complete | best val accuracy: {best_acc:.4f} "
      f"| held-out test accuracy: {final_test_acc:.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[RM] initialized trunk from: sft_model
[RM] val accuracy before training: 0.5333
[RM] epoch 0 step 0/68 loss 0.6303 batch_acc 0.750
[RM] epoch 0 step 50/68 loss 1.7700 batch_acc 0.250
[RM] epoch 0 done | train_loss 1.0666 train_acc 0.511 | val_acc 0.6333
[RM] new best val_acc 0.6333 -> saved reward_model.pt
[RM] epoch 1 step 0/68 loss 1.0547 batch_acc 0.500
[RM] epoch 1 step 50/68 loss 0.9733 batch_acc 0.250
[RM] epoch 1 done | train_loss 0.8851 train_acc 0.489 | val_acc 0.7000
[RM] new best val_acc 0.7000 -> saved reward_model.pt
[RM] epoch 2 step 0/68 loss 0.8876 batch_acc 0.750
[RM] epoch 2 step 50/68 loss 0.6184 batch_acc 0.500
[RM] epoch 2 done | train_loss 0.7131 train_acc 0.574 | val_acc 0.7333
[RM] new best val_acc 0.7333 -> saved reward_model.pt
[RM] training complete | best val accuracy: 0.7333 | held-out test accuracy: 0.6400


In [8]:
# ===== CELL 6: Stage 3 — PPO against RM with KL penalty to frozen SFT ref =====
import json, random, numpy as np, torch, torch.nn as nn
from transformers import AutoModelForCausalLM

class PolicyWithValue(nn.Module):
    def __init__(self, path):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(path)
        h = self.llm.config.hidden_size
        self.v_head = nn.Linear(h, 1, bias=False)
        nn.init.normal_(self.v_head.weight, std=1.0/np.sqrt(h+1))
    def forward(self, input_ids, attention_mask):
        out = self.llm(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        value = self.v_head(out.hidden_states[-1]).squeeze(-1)
        return out.logits, value
    def generate(self, *a, **k):
        return self.llm.generate(*a, **k)

# reload frozen reward model
ckpt = torch.load("reward_model.pt", map_location=device)
reward_model = RewardModel(ckpt["model_name"]).to(device).eval()
reward_model.load_state_dict(ckpt["state_dict"])
for p in reward_model.parameters(): p.requires_grad_(False)

# policy initialized from SFT; frozen SFT copy as KL reference
policy = PolicyWithValue("sft_model").to(device)
ref = AutoModelForCausalLM.from_pretrained("sft_model").to(device).eval()
for p in ref.parameters(): p.requires_grad_(False)

tokenizer.padding_side = "left"   # left-pad so generation continues correctly

prompts = [json.loads(l)["prompt"] for l in open("hh_rlhf_instructgpt/ppo.jsonl")]
random.seed(0); random.shuffle(prompts)

# hyperparameters
BATCH, MINI, PPO_EPOCHS = 8, 4, 1      # PPO_EPOCHS = inner reuse of each rollout batch
MAX_PROMPT_LEN, MAX_NEW  = 200, 40
BETA, GAMMA, LAM         = 0.5, 1.0, 0.95
CLIP, CLIP_V, VF         = 0.2, 0.2, 0.1
LR                       = 1e-6
NUM_EPOCHS               = 3            # full sweeps over the PPO prompt set

opt = torch.optim.AdamW(policy.parameters(), lr=LR)
gen_kwargs = dict(do_sample=True, top_k=0, top_p=1.0, max_new_tokens=MAX_NEW,
                  pad_token_id=tokenizer.pad_token_id)

def masked_mean(x, m): return (x*m).sum()/m.sum().clamp(min=1)
def masked_var(x, m):
    mu = masked_mean(x, m); return masked_mean((x-mu)**2, m)
def masked_whiten(x, m):
    mu, var = masked_mean(x, m), masked_var(x, m)
    return (x-mu)*torch.rsqrt(var+1e-8)
def logprobs_from(logits, labels):
    lp = torch.log_softmax(logits, -1)
    return torch.gather(lp, 2, labels.unsqueeze(-1)).squeeze(-1)

n_batches = max(1, len(prompts)//BATCH)
total_rollouts = NUM_EPOCHS * n_batches
# Each rollout batch triggers PPO_EPOCHS * ceil(BATCH/MINI) optimizer updates.
opt_per_rollout = PPO_EPOCHS * ((BATCH + MINI - 1) // MINI)
print(f"PPO: {NUM_EPOCHS} epochs x {n_batches} rollout batches = {total_rollouts} rollouts "
      f"({total_rollouts * opt_per_rollout} optimizer updates total)")

rollout_i = 0        # counts rollout batches (one fresh generation each)
opt_step  = 0        # counts optimizer.step() calls
for epoch in range(NUM_EPOCHS):
    random.shuffle(prompts)                       # reshuffle prompts each epoch
    ep_rm = ep_kl = 0.0                            # epoch-level running metrics
    for bi in range(n_batches):
        chunk = prompts[bi*BATCH : bi*BATCH + BATCH]
        enc = tokenizer(chunk, truncation=True, max_length=MAX_PROMPT_LEN,
                        padding=True, return_tensors="pt").to(device)
        plen = enc["input_ids"].size(1)

        # ---- rollout ----
        policy.eval()
        with torch.no_grad():
            full = policy.generate(input_ids=enc["input_ids"],
                                   attention_mask=enc["attention_mask"], **gen_kwargs)
        full_mask = (full != tokenizer.pad_token_id).long()
        resp_mask = torch.zeros_like(full_mask); resp_mask[:, plen:] = full_mask[:, plen:]

        with torch.no_grad():
            scores = reward_model(full, full_mask)                 # scalar per sequence
            logits, values = policy(full, full_mask)
            ref_logits = ref(input_ids=full, attention_mask=full_mask).logits
            labels = full[:, 1:]
            logp     = logprobs_from(logits[:, :-1], labels)
            ref_logp = logprobs_from(ref_logits[:, :-1], labels)
            values = values[:, :-1]
            mask = resp_mask[:, 1:].float()

            kl = logp - ref_logp
            rewards = -BETA * kl * mask                             # per-token KL penalty
            # Place RM scalar at the LAST RESPONSE TOKEN of each sequence.
            # The old code used `mask.sum(1)-1` (a token COUNT) as an index; with
            # left-padded prompts and right-padded short responses that count does
            # not equal the token's position and can land on a masked token,
            # silently dropping the reward. Instead find the highest index where
            # the (shifted) response mask is 1, which is robust to padding on
            # either side.
            T = mask.size(1)
            pos = torch.arange(T, device=device).unsqueeze(0)      # (1,T)
            last = (pos * mask.long()).argmax(dim=1)               # last response idx
            b = torch.arange(full.size(0), device=device)
            rewards[b, last] += scores                             # RM scalar at final token
            rewards = rewards * mask                               # keep reward on response region
            values = values * mask

            # GAE
            adv = torch.zeros_like(rewards); lastgae = 0.0; T = rewards.size(1)
            for t in reversed(range(T)):
                nextval = values[:, t+1] if t < T-1 else 0.0
                delta = rewards[:, t] + GAMMA*nextval - values[:, t]
                lastgae = delta + GAMMA*LAM*lastgae
                adv[:, t] = lastgae

            # returns = value-head target: must come from RAW advantages
            returns = adv + values
            # whiten a separate copy for the POLICY loss only
            adv = masked_whiten(adv, mask) * mask
        old_logp = logp.detach()

        # ---- PPO update ----
        policy.train()
        B = full.size(0)
        for _ in range(PPO_EPOCHS):
            idx = np.random.permutation(B)
            for s in range(0, B, MINI):
                mb = idx[s:s+MINI]
                lg, vv = policy(full[mb], full_mask[mb])
                lp = logprobs_from(lg[:, :-1], full[mb][:, 1:])
                vp = vv[:, :-1]; m = mask[mb]

                ratio = torch.exp(lp - old_logp[mb])
                pg1 = -adv[mb]*ratio
                pg2 = -adv[mb]*torch.clamp(ratio, 1-CLIP, 1+CLIP)
                pg_loss = masked_mean(torch.max(pg1, pg2), m)

                v_clip = values[mb] + torch.clamp(vp - values[mb], -CLIP_V, CLIP_V)
                v_loss = 0.5*masked_mean(torch.max((vp-returns[mb])**2, (v_clip-returns[mb])**2), m)

                loss = pg_loss + VF*v_loss
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
                opt.step()
                opt_step += 1

        # ---- logging ----
        ep_rm += scores.mean().item()
        ep_kl += masked_mean(kl, mask).item()
        rollout_i += 1
        if rollout_i % 10 == 0 or rollout_i == total_rollouts:
            print(f"[PPO] epoch {epoch} | rollout {rollout_i}/{total_rollouts} "
                  f"| opt_step {opt_step} | RM {scores.mean().item():.3f} "
                  f"| KL {masked_mean(kl, mask).item():.3f} "
                  f"| pg {pg_loss.item():.4f} | vf {v_loss.item():.4f}")

    print(f"[PPO] === epoch {epoch} done | mean RM {ep_rm/n_batches:.3f} | "
          f"mean KL {ep_kl/n_batches:.3f} ===")

policy.llm.save_pretrained("ppo_model"); tokenizer.save_pretrained("ppo_model")
# Save the value head too, so PPO training can be resumed (the LM save_pretrained
# only stores the transformer + LM head, not the separate v_head).
torch.save(policy.v_head.state_dict(), "ppo_model/value_head.pt")
print("Saved ppo_model/ (policy LM + value_head.pt)")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PPO: 3 epochs x 18 rollout batches = 54 rollouts (108 optimizer updates total)
[PPO] epoch 0 | rollout 10/54 | opt_step 20 | RM -2.135 | KL -0.027 | pg 2.6971 | vf 19.7268
[PPO] === epoch 0 done | mean RM -1.293 | mean KL -0.020 ===
[PPO] epoch 1 | rollout 20/54 | opt_step 40 | RM -1.504 | KL -0.009 | pg -0.4890 | vf 7.6309
[PPO] epoch 1 | rollout 30/54 | opt_step 60 | RM -2.680 | KL -0.029 | pg -0.1415 | vf 19.1893
[PPO] === epoch 1 done | mean RM -1.255 | mean KL -0.082 ===
[PPO] epoch 2 | rollout 40/54 | opt_step 80 | RM 0.651 | KL -0.169 | pg 0.0023 | vf 5.5816
[PPO] epoch 2 | rollout 50/54 | opt_step 100 | RM -1.557 | KL -0.097 | pg 0.5195 | vf 49.0979
[PPO] epoch 2 | rollout 54/54 | opt_step 108 | RM 0.111 | KL -0.281 | pg 0.2544 | vf 35.2355
[PPO] === epoch 2 done | mean RM -1.305 | mean KL -0.134 ===


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved ppo_model/ (policy LM + value_head.pt)


In [9]:
# ===== CELL 7: eval SFT vs PPO on HELD-OUT test prompts =====
import json, torch
from transformers import AutoModelForCausalLM

tokenizer.padding_side = "left"
sft_only = AutoModelForCausalLM.from_pretrained("sft_model").to(device).eval()
ppo_llm  = policy.llm.eval()

# held-out prompts the models never trained on.
# Use a real sample (not 5) so the RM win-rate is meaningful; N_SHOW are printed.
N_EVAL, N_SHOW = 100, 5
eval_prompts = [json.loads(l)["prompt"]
                for l in open("hh_rlhf_instructgpt/eval_prompts.jsonl")][:N_EVAL]

@torch.no_grad()
def score_seq(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(device)
    mask = torch.ones_like(enc["input_ids"])
    return reward_model(enc["input_ids"], mask).item()

@torch.no_grad()
def generate(model, text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=200).to(device)
    out = model.generate(**enc, max_new_tokens=40, do_sample=True, top_p=0.9,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].size(1):], skip_special_tokens=True).strip()

sft_scores, ppo_scores, ppo_wins = [], [], 0
for i, pr in enumerate(eval_prompts):
    text = format_prompt_for_gen(pr)
    sft_resp = generate(sft_only, text)
    ppo_resp = generate(ppo_llm, text)
    s_sft = score_seq(text + " " + sft_resp)
    s_ppo = score_seq(text + " " + ppo_resp)
    sft_scores.append(s_sft); ppo_scores.append(s_ppo)
    ppo_wins += int(s_ppo > s_sft)
    if i < N_SHOW:
        print("="*70)
        print("PROMPT:", pr[-200:])
        print(f"-- SFT [RM {s_sft:.3f}]:", sft_resp)
        print(f"-- PPO [RM {s_ppo:.3f}]:", ppo_resp)

import numpy as _np
print("\n" + "="*70)
print(f"AGGREGATE over {len(eval_prompts)} held-out prompts")
print(f"  mean RM  — SFT: {_np.mean(sft_scores):.3f}  |  PPO: {_np.mean(ppo_scores):.3f}")
print(f"  PPO win-rate vs SFT (RM-judged): {ppo_wins}/{len(eval_prompts)} "
      f"= {ppo_wins/len(eval_prompts):.1%}")
print("  (RM-judged win-rate is optimistic — PPO optimized against this same RM.)")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PROMPT: pattern.  A light that flashes repeatedly, as in a strobe light, causes a fluorescent material inside the tube to glow, like you see in a traffic light.

Human: Are they similar to a fluorescent bulb?
-- SFT [RM -4.009]: They’re different from any other type of light source, in that they have more electrons in them than a typical fluorescent bulb.  Another major difference is in the structure of the tubes.
-- PPO [RM -4.779]: They’re different.  One common distinction is that they’re much more expensive than a traditional fluorescing type of light source.  A typical fluorescent bulb uses a zinc-pl
PROMPT: Human: What do I do if I crash into someone's car and nobody is around?
-- SFT [RM -5.400]: You should think about it: if you think someone might be around, it might be safer to slow down and call 911. If your car is being driven erratically, you should try to drive in
-- PPO [RM -10.856]: Well, you might be thinking, "Well, here's a simple trick to get around safely:’
PROMPT: